# 5200 Final Exam — Mock Set 1 (with Answer Code)
## Dataset: eBay iPhone Auctions (`eBayClean.csv`)

### Exam Rules
- **Time**: 1 hr 50 min | **48 questions** (26 substantive + code pairs + setup)
- `alpha = 0.05` | `random_state = 617` | Do NOT round | No commas | Lead with 0
- Open-book; no communication with others

### Modules Covered
Logistic Regression · Feature Selection · Trees · Ensemble Models · SVM

### Variables
| Variable | Description |
|---|---|
| `sold` | **Outcome**: 1 = sold, 0 = not sold |
| `startprice` | Starting auction price |
| `biddable` | Whether auction allows bids |
| `condition` | Phone condition |
| `cellular` | Carrier type |
| `carrier` | Specific carrier |
| `color` | Phone color |
| `storage` | Storage capacity |
| `productline` | iPhone model |
| `noDescription` | 1 = no description |
| `charCountDescription` | Characters in description |
| `upperCaseDescription` | Uppercase chars in description |
| `startprice_99end` | Price ends in .99 |

---
## ❓ Q1. [Dataset Confirmation]
> Read in `eBayClean.csv`. How many rows and columns does the dataset have?

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('eBayClean.csv')
print(data.shape)   # → (rows, cols)

---
## ❓ Q2. [Setup]
> Assign `sold` to `y`. Assign all other features (except `UniqueID`) to `X`.
> Create `y_binned = pd.cut(y, bins=[-1,0,1])` for stratification.
> Split 70% train / 30% test, stratified, `random_state=617`.
> **What is the percentage of listings sold in the train sample?** (No % sign)

In [ ]:
from sklearn.model_selection import train_test_split

y = data.sold
X = data.drop(['UniqueID', 'sold'], axis=1)

y_binned = pd.cut(y, bins=[-1, 0, 1])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=617, stratify=y_binned)

# % sold in train (no % sign)
print(y_train.mean() * 100)

---
## Section 1: Logistic Regression — StatsModels
*(Unless stated otherwise, use the TRAIN sample)*

---
## ❓ Q3.
> Compare the average `startprice` for listings that sold vs those that did not.
> **What is the average `startprice` for listings that DID sell (sold=1)?**

In [ ]:
train = X_train_raw.copy()
train['sold'] = y_train.values

print(train.groupby('sold')['startprice'].mean())
# Answer: row where sold=1

---
## ❓ Q4.
> Examine the effect of `startprice` on `sold` using logistic regression. Call this `model1`.
> **Is the coefficient of `startprice` statistically significant at alpha=0.05?**

In [ ]:
from statsmodels.formula.api import logit

model1 = logit('sold ~ startprice', data=train).fit()
print(model1.summary())

# Check significance:
print('\np-value of startprice:', model1.pvalues['startprice'])
print('Significant:', model1.pvalues['startprice'] < 0.05)

---
## ❓ Q5.
> Based on `model1`, what is the predicted probability that the **first listing in train** will sell?

In [ ]:
# Predicted probability for first row in train
model1.predict(train.iloc[[0]])

---
## ❓ Q6.
> Based on `model1`, what is the predicted probability of selling for a listing with `startprice = 300`?

In [ ]:
model1.predict(pd.DataFrame({'startprice': [300]}))

---
## ❓ Q7.
> Based on `model1`, if `startprice` goes up by one dollar, what is the **percent change** in the likelihood (odds) of a listing selling?
> Hint: `(exp(coef) - 1) * 100`

In [ ]:
pct_change = (np.exp(model1.params['startprice']) - 1) * 100
print('Percent change in odds per $1 increase:', pct_change)

---
## ❓ Q8.
> Construct a logistic regression to predict `sold` from `condition`. Call this `model2`.
> **What is the coefficient for the 'Used' condition level** (vs. the reference)?

In [ ]:
model2 = logit('sold ~ condition', data=train).fit()
print(model2.summary())
print('\nCoefficients:')
print(model2.params)

---
## ❓ Q9.
> Construct a logistic regression predicting `sold` using ALL available features (statsmodels). Call this `model3`.
> **What is the Pseudo R-squared (McFadden) value?**

In [ ]:
# Build formula from all features
features = [c for c in train.columns if c != 'sold']
formula = 'sold ~ ' + ' + '.join(features)

model3 = logit(formula, data=train).fit(maxiter=200)
print('Pseudo R²:', model3.prsquared)

---
## ❓ Q10.
> Based on `model3`, which of the following are statistically significant?
> Check: `startprice`, `charCountDescription`, `upperCaseDescription`

In [ ]:
# All p-values
print(model3.pvalues.sort_values())

# Significant at 0.05:
print('\nSignificant (p<0.05):')
print(model3.pvalues[model3.pvalues < 0.05])

---
## Section 2: Logistic Regression — Scikit-Learn

---
## ❓ Q11.
> Dummy-encode categorical variables using `pd.get_dummies(drop_first=True)`.
> Do this for both train and test; align test columns to train.
> **What is the mean value of the dummy variable for `biddable`?**

In [ ]:
cat_cols = ['biddable','condition','cellular','carrier','color',
            'storage','productline','noDescription','startprice_99end']

X_tr = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True).astype(float)
X_te = pd.get_dummies(X_test_raw,  columns=cat_cols, drop_first=True).astype(float)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

print('Columns:', list(X_tr.columns))

# Mean of biddable dummy (find the column name)
biddable_col = [c for c in X_tr.columns if 'biddable' in c.lower()]
print('Biddable columns:', biddable_col)
for col in biddable_col:
    print(f'  Mean of {col}:', X_tr[col].mean())

---
## ❓ Q12.
> Fit `LogisticRegression(max_iter=10000, random_state=617)`.
> **What is the predicted CLASS for the last observation in the train sample?**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=10000, random_state=617)
lr.fit(X_tr, y_train)

pred_train = lr.predict(X_tr)
pred_test  = lr.predict(X_te)

# Predicted class for LAST observation in train
print('Predicted class (last train obs):', pred_train[-1])

---
## ❓ Q13.
> **What is the Precision for the sold=1 class in the TRAIN sample?**
> (Precision = TP / (TP + FP): minimizes false alarms)

In [ ]:
from sklearn.metrics import precision_score, recall_score, classification_report

print('Train Precision:', precision_score(y_train, pred_train))
print('\nFull report (train):')
print(classification_report(y_train, pred_train))

---
## ❓ Q14.
> **What is the Sensitivity (Recall) for the sold=1 class in the TEST sample?**
> (Sensitivity = TP / (TP + FN): minimizes missed sales)

In [ ]:
print('Test Sensitivity (Recall):', recall_score(y_test, pred_test))

---
## ❓ Q15.
> **What is the AUC for the TRAIN sample?**

In [ ]:
from sklearn.metrics import roc_auc_score

proba_train = lr.predict_proba(X_tr)[:, 1]
print('Train AUC:', roc_auc_score(y_train, proba_train))

---
## ❓ Q16.
> **What is the AUC for the TEST sample?**

In [ ]:
proba_test = lr.predict_proba(X_te)[:, 1]
print('Test AUC:', roc_auc_score(y_test, proba_test))

---
## Section 3: Decision Trees

---
## ❓ Q17.
> Fit a Classification Tree predicting `sold` using only `startprice`.
> Set `max_depth=2, random_state=617`. Plot the tree.
> **What is the predicted CLASS for a listing with `startprice=150`?**

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

tree1 = DecisionTreeClassifier(max_depth=2, random_state=617)
tree1.fit(X_tr[['startprice']], y_train)

plt.figure(figsize=(12, 5))
plot_tree(tree1, feature_names=['startprice'], filled=True, class_names=['Not Sold','Sold'])
plt.show()

# Predict for startprice=150
print('Predicted class (startprice=150):', tree1.predict([[150]])[0])

---
## ❓ Q18.
> Fit a Classification Tree using ALL dummy-encoded features.
> Set `max_depth=4, random_state=617`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.metrics import accuracy_score

tree2 = DecisionTreeClassifier(max_depth=4, random_state=617)
tree2.fit(X_tr, y_train)

print('Train accuracy:', accuracy_score(y_train, tree2.predict(X_tr)))

---
## ❓ Q19.
> For the `max_depth=4` tree, **what is the TEST accuracy?**

In [ ]:
print('Test accuracy:', accuracy_score(y_test, tree2.predict(X_te)))

---
## ❓ Q20.
> Tune the tree using `GridSearchCV` with 5-fold CV.
> Tune `max_depth` over `[2, 4, 6, 8, 10]`.
> **What is the optimal `max_depth`?**

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=617),
    {'max_depth': [2, 4, 6, 8, 10]},
    cv=5, n_jobs=-1
)
grid_tree.fit(X_tr, y_train)

print('Best params:', grid_tree.best_params_)
print('Best CV score:', grid_tree.best_score_)

---
## Section 4: Ensemble Models

---
## ❓ Q21.
> Fit a **Bagging Classifier** with 100 estimators, `random_state=617`.
> Use all dummy-encoded features.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.ensemble import BaggingClassifier

bag = BaggingClassifier(n_estimators=100, random_state=617)
bag.fit(X_tr, y_train)

print('Bagging Train accuracy:', bag.score(X_tr, y_train))
print('Bagging Test accuracy: ', bag.score(X_te, y_test))

---
## ❓ Q22.
> Fit a **Random Forest** with 100 trees, `oob_score=True, random_state=617`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=617)
rf.fit(X_tr, y_train)

print('RF Train accuracy:', rf.score(X_tr, y_train))
print('RF Test accuracy: ', rf.score(X_te, y_test))
print('RF OOB score:     ', rf.oob_score_)

---
## ❓ Q23.
> Based on the Random Forest, **which feature is MOST IMPORTANT?**

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_tr.columns)
print(importances.sort_values(ascending=False))
print('\nMost important feature:', importances.idxmax())

---
## ❓ Q24.
> Fit a **Gradient Boosting Classifier** with 100 trees, `random_state=617`.
> **What is the TEST accuracy?**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=617)
gb.fit(X_tr, y_train)

print('GB Train accuracy:', gb.score(X_tr, y_train))
print('GB Test accuracy: ', gb.score(X_te, y_test))

---
## Section 5: Support Vector Machines

---
## ❓ Q25.
> Use numeric features only: `startprice`, `charCountDescription`, `upperCaseDescription`.
> Scale with `StandardScaler` (fit on train, transform both).
> Fit `LinearSVC(C=1, random_state=617)`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

numeric_cols = ['startprice', 'charCountDescription', 'upperCaseDescription']
X_svm_tr_raw = X_train_raw[numeric_cols]
X_svm_te_raw = X_test_raw[numeric_cols]

scaler = StandardScaler()
X_svm_tr = scaler.fit_transform(X_svm_tr_raw)  # fit ONLY on train
X_svm_te = scaler.transform(X_svm_te_raw)       # only transform test

svm_lin = LinearSVC(C=1, random_state=617)
svm_lin.fit(X_svm_tr, y_train)

print('LinearSVC Train accuracy:', svm_lin.score(X_svm_tr, y_train))
print('LinearSVC Test accuracy: ', svm_lin.score(X_svm_te, y_test))

---
## ❓ Q26.
> Tune an RBF SVM using `GridSearchCV` with 5-fold CV.
> `C = [0.1, 1, 10, 100]`, `gamma = [0.001, 0.01, 0.1, 1]`.
> **What is the best C and gamma? What is the TEST accuracy of the best model?**

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1]}
grid_svm = GridSearchCV(SVC(kernel='rbf', random_state=617), param_grid, cv=5, n_jobs=-1)
grid_svm.fit(X_svm_tr, y_train)

print('Best params:        ', grid_svm.best_params_)
print('Best CV score:      ', grid_svm.best_score_)
print('Tuned SVM Train acc:', grid_svm.score(X_svm_tr, y_train))
print('Tuned SVM Test acc: ', grid_svm.score(X_svm_te, y_test))

---
# 📋 Quick Reference — Code Patterns

## Logistic Regression (StatsModels)
```python
from statsmodels.formula.api import logit
model = logit('y ~ x1 + x2', data=train).fit()
model.prsquared              # Pseudo R²
model.pvalues                # p-values
model.params                 # coefficients
model.predict(train)         # probabilities (not classes)
(np.exp(model.params['x']) - 1) * 100   # % change in odds
```

## Logistic Regression (Sklearn)
```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, recall_score,
                             precision_score, classification_report)
lr = LogisticRegression(max_iter=10000, random_state=617)
lr.fit(X_tr, y_tr)
pred = lr.predict(X_te)                     # class labels
proba = lr.predict_proba(X_te)[:, 1]        # P(Y=1)
roc_auc_score(y_te, proba)                  # AUC
recall_score(y_te, pred)                    # Sensitivity
precision_score(y_te, pred)                 # Precision
```

## Decision Tree (Classification)
```python
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = DecisionTreeClassifier(max_depth=4, random_state=617)
tree.fit(X_tr, y_tr)
accuracy_score(y_te, tree.predict(X_te))
```

## Ensemble
```python
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               GradientBoostingClassifier)
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=617)
rf.oob_score_
pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values(ascending=False)
```

## SVM
```python
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)   # fit on train only!
X_te = scaler.transform(X_te_raw)
```

## ⚠️ Key Exam Rules
| Rule | Detail |
|---|---|
| Scaling for SVM | ALWAYS fit scaler on train only |
| Pseudo R² | `model.prsquared` (not `rsquared`) |
| Sensitivity | `recall_score()` = TP/(TP+FN) |
| Precision | `precision_score()` = TP/(TP+FP) |
| AUC | `roc_auc_score(y_true, predict_proba[:,1])` |
| OOB score | `rf.oob_score_` (after `oob_score=True`) |
| Feature importance | `rf.feature_importances_` with column names |